In [1]:
import requests
import time
import pandas as pd
from datetime import datetime
import os
import yaml
from tqdm import tqdm  # New import for the progress bar

In [2]:
def datetime_to_ms(date_str):
    """Convert a datetime string to millisecond timestamp."""
    # Support both formats: with and without fractional seconds
    if '.' in str(date_str):
        dt_obj = datetime.strptime(str(date_str), "%Y-%m-%d %H:%M:%S.%f")
    else:
        dt_obj = datetime.strptime(str(date_str), "%Y-%m-%d %H:%M:%S")
    return int(dt_obj.timestamp() * 1000)

def load_config(config_path="config.yaml"):
    """Load the YAML configuration file."""
    with open(config_path, 'r') as file:
        return yaml.safe_load(file)

def fetch_and_save_binance_data(config_path="config.yaml"):
    """
    Fetch Binance data, append to existing data (incremental load)[cite: 78], 
    and save securely to Google Drive[cite: 79].
    """
    # Load configuration
    config = load_config(config_path)
    symbol = config['symbol']
    timeframe = config['timeframe']
    start_date_cfg = config['start_date']
    end_date_cfg = config['end_date']
    save_dir = config['save_dir']

    # Use Data API to prevent HTTP 451 (Geo-blocking) errors [cite: 71]
    base_url = "https://data-api.binance.vision/api/v3/klines"
    limit = 1000  
    
    end_ts = datetime_to_ms(end_date_cfg)
    
    # Standard Binance columns definition
    columns = [
        'Open_time', 'Open', 'High', 'Low', 'Close', 'Volume',
        'Close_time', 'Quote_asset_volume', 'Number_of_trades',
        'Taker_buy_base_asset_volume', 'Taker_buy_quote_asset_volume', 'Ignore'
    ]
    
    # Create the directory if it does not exist
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    file_path = os.path.join(save_dir, f"{symbol}_{timeframe}_data.csv")
    
    main_df = pd.DataFrame()
    start_ts = None

    # ==========================================
    # INCREMENTAL LOAD MECHANISM
    # ==========================================
    if os.path.exists(file_path):
        print(f"📂 Found existing data file at: {file_path}")
        existing_df = pd.read_csv(file_path)
        
        if not existing_df.empty:
            # Extract the close time of the last row [cite: 86]
            last_close_time_str = existing_df.iloc[-1]['Close_time']
            # Convert to ms and add 1ms to avoid fetching the exact same candle [cite: 87]
            start_ts = datetime_to_ms(last_close_time_str) + 1
            main_df = existing_df
            print(f"Resuming download from timestamp: {last_close_time_str}")
        else:
            print("File exists but is empty. Using start_date from config.")
            start_ts = datetime_to_ms(start_date_cfg)
    else:
        print("Creating a completely new file. Using start_date from config.")
        start_ts = datetime_to_ms(start_date_cfg)

    # Check if data is already fully updated [cite: 88]
    if start_ts >= end_ts:
        print("✅ Data is already fully updated up to the requested end_date. No further download needed.")
        return main_df

    print(f"🚀 Starting data collection...")
    
    # ==========================================
    # DATA FETCHING LOOP WITH PROGRESS BAR
    # ==========================================
    # Calculate the total duration for the progress bar
    total_duration = end_ts - start_ts
    
    # Initialize the progress bar
    # We use 'ms' as the unit, but tqdm will display the percentage correctly
    pbar = tqdm(total=total_duration, desc=f"Downloading {symbol}", unit="ms", 
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} ms [{elapsed}<{remaining}, {rate_fmt}]")

    current_ts = start_ts

    while current_ts < end_ts:
        params = {
            "symbol": symbol,
            "interval": timeframe,
            "startTime": current_ts,
            "endTime": end_ts,
            "limit": limit
        }
        
        max_retries = 5
        retry_count = 0
        success = False
        data = []
        
        # Defensive Mechanism: Exponential Backoff [cite: 72]
        while retry_count < max_retries and not success:
            try:
                response = requests.get(base_url, params=params, timeout=15)
                response.raise_for_status() 
                data = response.json()
                success = True 
                
            except Exception as e:
                print(f"\n⚠️ [Error] {e}. Retrying ({retry_count+1}/{max_retries})...")
                retry_count += 1
                time.sleep(5 * retry_count) 
                
        if not success or not data:
            print("\n🚨 Failed to fetch data after multiple attempts or no more data available.")
            break
            
        # Process the newly fetched batch into a DataFrame
        batch_df = pd.DataFrame(data, columns=columns)
        batch_df['Open_time'] = pd.to_datetime(batch_df['Open_time'], unit='ms')
        batch_df['Close_time'] = pd.to_datetime(batch_df['Close_time'], unit='ms')
        
        # Convert string values to floats
        numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Quote_asset_volume', 
                        'Taker_buy_base_asset_volume', 'Taker_buy_quote_asset_volume']
        batch_df[numeric_cols] = batch_df[numeric_cols].apply(pd.to_numeric, axis=1)
        batch_df = batch_df.drop(columns=['Ignore'])
        
        # Append the new batch to the main DataFrame
        main_df = pd.concat([main_df, batch_df], ignore_index=True)
        
        # Update timestamps and progress bar
        last_kline_close_time = data[-1][6]
        
        # Calculate how much time we advanced in this iteration
        step_duration = (last_kline_close_time + 1) - current_ts
        pbar.update(step_duration) # Update the visual progress bar
        
        current_ts = last_kline_close_time + 1
        
        # Sleep slightly to respect server rate limits
        time.sleep(0.2)
        
        # Save to Google Drive safely after each successful request
        main_df.to_csv(file_path, index=False)

    # Close the progress bar
    pbar.close()

    print("\n=========================================")
    print(f"🎉 UPDATE COMPLETED! Total dataset size: {len(main_df)} candles.")
    print(f"📂 Data safely saved at: {file_path}")
    print("=========================================")
    
    return main_df

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    # Ensure your config.yaml is created at this path
    CONFIG_FILE_PATH = "/workspaces/Binance-Predict-Value-And-Trend/config.yaml"
    
    try:
        final_df = fetch_and_save_binance_data(config_path=CONFIG_FILE_PATH)
        display(final_df.tail()) # Display the last 5 rows to verify
    except FileNotFoundError:
        print("🚨 Error: config.yaml not found. Please create the configuration file before running.")

📂 Found existing data file at: /workspaces/Binance-Predict-Value-And-Trend/data/raw/BTCUSDT_1m_data.csv
Resuming download from timestamp: 2023-01-10 00:00:59.999
✅ Data is already fully updated up to the requested end_date. No further download needed.


,Open_time,Open,High,Low,Close,Volume,Close_time,Quote_asset_volume,Number_of_trades,Taker_buy_base_asset_volume,Taker_buy_quote_asset_volume
12956,2023-01-09 23:56:00,17189.86,17193.71,17189.86,17191.33,60.59261,2023-01-09 23:56:59.999,1.041708e+06,1993,30.53071,5.248899e+05
12957,2023-01-09 23:57:00,17191.33,17192.57,17190.03,17190.39,44.75544,2023-01-09 23:57:59.999,7.694088e+05,1548,22.54337,3.875600e+05
12958,2023-01-09 23:58:00,17190.39,17191.50,17184.94,17187.42,96.22677,2023-01-09 23:58:59.999,1.653931e+06,2473,51.14071,8.790027e+05
12959,2023-01-09 23:59:00,17187.42,17187.74,17178.00,17178.26,139.30909,2023-01-09 23:59:59.999,2.393451e+06,3024,50.45635,8.668796e+05
12960,2023-01-10 00:00:00,17179.04,17184.12,17178.19,17182.31,131.36468,2023-01-10 00:00:59.999,2.257004e+06,3521,74.00751,1.271546e+06


In [3]:
df = pd.read_csv("/workspaces/Binance-Predict-Value-And-Trend/data/raw/BTCUSDT_1m_data.csv")
df.head(10)

,Open_time,Open,High,Low,Close,Volume,Close_time,Quote_asset_volume,Number_of_trades,Taker_buy_base_asset_volume,Taker_buy_quote_asset_volume
0,2023-01-01 00:00:00,16541.77,16544.76,16538.45,16543.67,83.08143,2023-01-01 00:00:59.999,1.374269e+06,2687,40.18369,6.647060e+05
1,2023-01-01 00:01:00,16543.04,16544.41,16538.48,16539.31,80.45300,2023-01-01 00:01:59.999,1.330773e+06,2890,33.28428,5.505633e+05
2,2023-01-01 00:02:00,16539.31,16541.17,16534.52,16536.43,62.90197,2023-01-01 00:02:59.999,1.040248e+06,1930,20.73999,3.430094e+05
3,2023-01-01 00:03:00,16536.43,16537.28,16531.00,16533.65,115.71894,2023-01-01 00:03:59.999,1.913268e+06,2956,49.40758,8.169194e+05
4,2023-01-01 00:04:00,16534.12,16536.08,16527.51,16535.38,144.45369,2023-01-01 00:04:59.999,2.388081e+06,3795,76.52421,1.265080e+06
5,2023-01-01 00:05:00,16534.91,16537.80,16533.94,16536.70,53.58957,2023-01-01 00:05:59.999,8.861410e+05,2130,30.89856,5.109335e+05
6,2023-01-01 00:06:00,16536.76,16540.43,16534.20,16535.91,96.45571,2023-01-01 00:06:59.999,1.595134e+06,2348,42.82683,7.082578e+05
7,2023-01-01 00:07:00,16535.91,16539.69,16533.95,16534.67,56.72010,2023-01-01 00:07:59.999,9.379698e+05,2159,27.87678,4.610050e+05
8,2023-01-01 00:08:00,16533.98,16534.79,16528.00,16529.28,95.13552,2023-01-01 00:08:59.999,1.572652e+06,2619,48.51311,8.019371e+05
9,2023-01-01 00:09:00,16529.28,16529.52,16522.55,16526.67,89.28953,2023-01-01 00:09:59.999,1.475565e+06,2688,39.49913,6.527458e+05


In [4]:
import plotly.graph_objects as go
import pandas as pd

def plot_binance_candles(df):
    """
    Vẽ biểu đồ nến tương tác với phong cách giao diện Binance.
    """
    # Đảm bảo các cột định dạng đúng
    df['Open_time'] = pd.to_datetime(df['Open_time'])

    # Khởi tạo biểu đồ nến
    fig = go.Figure(data=[go.Candlestick(
        x=df['Open_time'],
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        increasing_line_color='#26a69a',  # Màu xanh của Binance
        decreasing_line_color='#ef5350'   # Màu đỏ của Binance
    )])

    # Tùy chỉnh giao diện (Layout) giống Binance
    fig.update_layout(
        title='Binance Price Chart',
        yaxis_title='Price (USDT)',
        xaxis_title='Date',
        template='plotly_dark',  # Nền tối
        xaxis_rangeslider_visible=False, # Ẩn thanh trượt bên dưới để tập trung biểu đồ
        paper_bgcolor='#131722',  # Màu nền giấy giống Binance
        plot_bgcolor='#131722',   # Màu nền biểu đồ
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # Thêm khả năng tương tác (phóng to, di chuyển)
    fig.show()

# Giả sử bạn đã có biến 'final_df' từ bước cào dữ liệu:
plot_binance_candles(df)